# Stellar Classification - Exploratory Data Analysis (EDA)

This notebook contains the exploratory data analysis for the Kaggle Playground Series S6E6: Predicting Stellar Class. The goal is to predict the `class` of astronomical objects (GALAXY, STAR, QSO) using photometric and positional data.

This notebook is tailored to run seamlessly both locally and on Kaggle. It dynamically resolves path configurations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot styling to match project standards (Viridis default)
sns.set_theme(style="whitegrid")
sns.set_palette("viridis")

## 1. Path Resolutions & Data Loading

In [ ]:
# Dynamic path resolution: Kaggle (competition input path) vs Local
if os.path.exists('/kaggle/input/competitions/playground-series-s6e6'):
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
else:
    DATA_DIR = '../data'

train_path = os.path.join(DATA_DIR, 'train.csv')
test_path = os.path.join(DATA_DIR, 'test.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"Data Directory in use: {DATA_DIR}")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

## 2. Target Class Distribution

Balanced Accuracy is our evaluation metric, which means class imbalance must be accounted for during training.

In [ ]:
counts = train['class'].value_counts()
pcts = train['class'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Count': counts, 'Percentage': pcts})
print(dist_df)

plt.figure(figsize=(6, 4))
sns.countplot(data=train, x='class', hue='class', palette='viridis', legend=False)
plt.title('Target Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Redshift Distribution by Target Class

Redshift is a critical cosmological parameter measuring how much light from an object is shifted toward longer wavelengths due to the expansion of the universe.

In [ ]:
print("--- Redshift Summary by Class ---")
print(train.groupby('class')['redshift'].describe())

plt.figure(figsize=(10, 5))
sns.histplot(data=train, x='redshift', hue='class', kde=True, bins=100, palette='viridis', multiple='stack')
plt.title('Redshift Distribution by Class')
plt.xlabel('Redshift')
plt.ylabel('Count')
plt.yscale('log') # Using log scale due to broad range and peak density differences
plt.tight_layout()
plt.show()

## 4. Astronomical Color Indices

Color indices represent the difference in magnitudes between different bands (SDSS u, g, r, i, z). They serve as proxy indicators for temperature, spectral energy distribution, and stellar/galaxy types.

In [ ]:
# Engineering color index differences
train['u_g'] = train['u'] - train['g']
train['g_r'] = train['g'] - train['r']
train['r_i'] = train['r'] - train['i']
train['i_z'] = train['i'] - train['z']

# Sample data for plotting to keep memory/runtime small
sample = train.sample(20000, random_state=42)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=sample, x='g_r', y='u_g', hue='class', palette='viridis', alpha=0.4, s=10)
plt.title('Color-Color Diagram: u-g vs g-r (Sample of 20k)')
plt.xlabel('g - r')
plt.ylabel('u - g')
plt.tight_layout()
plt.show()

## 5. Celestial Coordinates

declination (`delta`) and right ascension (`alpha`) represent position on the celestial sphere.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=sample, x='alpha', y='delta', hue='class', palette='viridis', alpha=0.3, s=8)
plt.title('Celestial Positions (Sample of 20k)')
plt.xlabel('Alpha (Right Ascension)')
plt.ylabel('Delta (Declination)')
plt.tight_layout()
plt.show()

## 6. Categorical Features vs Target Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=train, x='spectral_type', hue='class', palette='viridis', ax=axes[0])
axes[0].set_title('Spectral Type by Target Class')
axes[0].set_xlabel('Spectral Type')

sns.countplot(data=train, x='galaxy_population', hue='class', palette='viridis', ax=axes[1])
axes[1].set_title('Galaxy Population by Target Class')
axes[1].set_xlabel('Galaxy Population')

plt.tight_layout()
plt.show()